# Heart Disease Diagnostic Classification

###  Overview
The goal of this project is to build a machine learning system to predict the presence of heart disease in patients based on clinical features. We are comparing **Logistic Regression** (baseline) and **Random Forest** (complex) to determine which model offers the best balance of precision and recall.

###  Medical Logic & Constraints
In this diagnostic context, **Recall** is our most critical metric. Missing a sick patient (False Negative) is much more dangerous than a False Positive. This project implements:
* **Stratified Splitting:** Ensuring the proportion of sick vs. healthy patients is consistent across train/test sets.
* **Scikit-Learn Pipelines:** Preventing **Data Leakage** by strictly separating preprocessing from model training.
* **Class Weights:** Addressing any potential imbalance in the target variable.

###  Dataset Source
The data used is the **UCI Heart Disease Dataset**, containing 303 patient records with 14 clinical attributes (age, sex, chest pain type, cholesterol, etc.).

In [1]:
# All the necessary imports
# Core Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model Selection and pipelines
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Preprocessing 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation Metrics 
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

In [6]:
# Loading the dataset
url = "https://raw.githubusercontent.com/JWarmenhoven/ISLR-python/master/Notebooks/Data/Heart.csv"
df = pd.read_csv(url)
# put the csv file in the data folder to save it locally
df.to_csv("data/heart.csv", index=False)
print(df.shape)


(303, 15)


In [7]:
print(df.columns)
display(df.head())

Index(['Unnamed: 0', 'Age', 'Sex', 'ChestPain', 'RestBP', 'Chol', 'Fbs',
       'RestECG', 'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'AHD'],
      dtype='object')


,Unnamed: 0,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
0,1,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
1,2,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
2,3,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
3,4,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
4,5,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


In [8]:
df_set = df.copy()
df_set.drop(columns=["Unnamed: 0"], inplace=True)
#mapping target to 0 and 1
df_set["AHD"] = df_set["AHD"].map({"Yes": 1, "No": 0})

In [10]:
# General data overview
print("--- Data Info ---")
df_set.info()

display(df_set.sample(10))

print("\n--- Statistical Summary ---")
display(df_set.describe())

# Checking for missing values
print("\n--- Missing Values ---")
print(df_set.isnull().sum())

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        303 non-null    int64  
 1   Sex        303 non-null    int64  
 2   ChestPain  303 non-null    object 
 3   RestBP     303 non-null    int64  
 4   Chol       303 non-null    int64  
 5   Fbs        303 non-null    int64  
 6   RestECG    303 non-null    int64  
 7   MaxHR      303 non-null    int64  
 8   ExAng      303 non-null    int64  
 9   Oldpeak    303 non-null    float64
 10  Slope      303 non-null    int64  
 11  Ca         299 non-null    float64
 12  Thal       301 non-null    object 
 13  AHD        303 non-null    int64  
dtypes: float64(2), int64(10), object(2)
memory usage: 33.3+ KB


,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
139,51,1,nonanginal,125,245,1,2,166,0,2.4,2,0.0,normal,0
195,67,1,asymptomatic,100,299,0,2,125,1,0.9,2,2.0,normal,1
220,41,0,nonanginal,112,268,0,2,172,1,0.0,1,0.0,normal,0
181,56,0,asymptomatic,134,409,0,2,150,1,1.9,2,2.0,reversable,1
55,54,1,asymptomatic,124,266,0,2,109,1,2.2,2,1.0,reversable,1
73,65,1,asymptomatic,110,248,0,2,158,0,0.6,1,2.0,fixed,1
39,61,1,nonanginal,150,243,1,0,137,1,1.0,2,0.0,normal,0
228,54,1,asymptomatic,110,206,0,2,108,1,0.0,2,1.0,normal,1
183,59,1,typical,178,270,0,2,145,0,4.2,3,0.0,reversable,0
236,56,1,asymptomatic,130,283,1,2,103,1,1.6,3,0.0,reversable,1



--- Statistical Summary ---


,Age,Sex,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,AHD
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,303.000000
mean,54.438944,0.679868,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,0.458746
std,9.038662,0.467299,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,0.499120
min,29.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,48.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,0.000000
50%,56.000000,1.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,0.000000
75%,61.000000,1.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,1.000000
max,77.000000,1.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,1.000000



--- Missing Values ---
Age          0
Sex          0
ChestPain    0
RestBP       0
Chol         0
Fbs          0
RestECG      0
MaxHR        0
ExAng        0
Oldpeak      0
Slope        0
Ca           4
Thal         2
AHD          0
dtype: int64


In [11]:
# knowing the target distribution
print(df_set["AHD"].value_counts(normalize=True))

AHD
0    0.541254
1    0.458746
Name: proportion, dtype: float64


In [12]:
print(df_set["ChestPain"].value_counts())
print(df_set["Thal"].value_counts())

ChestPain
asymptomatic    144
nonanginal       86
nontypical       50
typical          23
Name: count, dtype: int64
Thal
normal        166
reversable    117
fixed          18
Name: count, dtype: int64


In [13]:
#splitting the data with target for stratification
X = df_set.drop("AHD", axis=1)
y = df_set["AHD"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)